# CP322 Assignment 1 — Q2: Decision Trees

This notebook classifies real vs. fake news headlines using `CountVectorizer` and `DecisionTreeClassifier`.
It performs the required 70%/15%/15% split, selects `max_depth` using validation accuracy, evaluates the selected model on the test set, and prints the first two layers of the selected tree.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier, export_text


## (a) Load data and extract word-count features


In [ ]:
def load_data():
    with open("Decision Tree-Dataset/real.txt", "r", encoding="utf-8") as file:
        real_headlines = [line.strip() for line in file if line.strip()]

    with open("Decision Tree-Dataset/fake.txt", "r", encoding="utf-8") as file:
        fake_headlines = [line.strip() for line in file if line.strip()]

    headlines = real_headlines + fake_headlines
    labels = np.array([1] * len(real_headlines) + [0] * len(fake_headlines))

    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(headlines)

    rng = np.random.default_rng(42)
    indices = rng.permutation(len(headlines))
    X = X[indices]
    labels = labels[indices]

    n = len(labels)
    train_end = int(0.70 * n)
    validation_end = int(0.85 * n)

    X_train, y_train = X[:train_end], labels[:train_end]
    X_validation, y_validation = X[train_end:validation_end], labels[train_end:validation_end]
    X_test, y_test = X[validation_end:], labels[validation_end:]

    print("Training examples:", len(y_train))
    print("Validation examples:", len(y_validation))
    print("Test examples:", len(y_test))

    return X_train, y_train, X_validation, y_validation, X_test, y_test, vectorizer


## (b) Select the best decision-tree depth


In [ ]:
def select_model(X_train, y_train, X_validation, y_validation, X_test, y_test):
    depths = [2, 5, 10, 20, 50, 100, 150, 200, 300]
    validation_accuracies = []

    for depth in depths:
        model = DecisionTreeClassifier(
            criterion="entropy", max_depth=depth, random_state=42
        )
        model.fit(X_train, y_train)
        predictions = model.predict(X_validation)

        # Validation accuracy computed directly from predictions.
        accuracy = np.mean(predictions == y_validation)
        validation_accuracies.append(accuracy)
        print("max_depth =", depth, "validation accuracy:", accuracy)

    best_index = np.argmax(validation_accuracies)
    best_depth = depths[best_index]

    best_model = DecisionTreeClassifier(
        criterion="entropy", max_depth=best_depth, random_state=42
    )
    best_model.fit(X_train, y_train)

    test_predictions = best_model.predict(X_test)
    test_accuracy = np.mean(test_predictions == y_test)

    print()
    print("Best max_depth:", best_depth)
    print("Actual depth of the trained tree:", best_model.get_depth())
    print("Test accuracy:", test_accuracy)

    plt.figure(figsize=(7, 5))
    plt.plot(depths, validation_accuracies, marker="o")
    plt.xlabel("max_depth")
    plt.ylabel("Validation accuracy")
    plt.title("Validation accuracy vs. max_depth")
    plt.grid(True)
    plt.show()

    return best_model, best_depth, validation_accuracies, test_accuracy


## (c) Extract the first two layers of the best tree


In [ ]:
def visualize_tree(best_model, vectorizer):
    feature_names = vectorizer.get_feature_names_out()
    tree_text = export_text(
        best_model,
        feature_names=list(feature_names),
        max_depth=1
    )
    print("First Two Layers of Best Decision Tree:")
    print(tree_text)
    return tree_text


## Run Q2


In [ ]:
X_train, y_train, X_validation, y_validation, X_test, y_test, vectorizer = load_data()
best_model, best_depth, validation_accuracies, test_accuracy = select_model(
    X_train, y_train, X_validation, y_validation, X_test, y_test
)
tree_text = visualize_tree(best_model, vectorizer)


## Saved results from the completed run

The following output records the results supplied from the completed Q2 run. Re-running the cell above with the assignment dataset will regenerate the results and the Part (c) tree text.


In [1]:
# Saved console results from the completed run


Training examples: 2286
Validation examples: 490
Test examples: 490
max_depth = 2 validation accuracy: 0.6
max_depth = 5 validation accuracy: 0.7183673469387755
max_depth = 10 validation accuracy: 0.7244897959183674
max_depth = 20 validation accuracy: 0.7428571428571429
max_depth = 50 validation accuracy: 0.7489795918367347
max_depth = 100 validation accuracy: 0.773469387755102
max_depth = 150 validation accuracy: 0.773469387755102
max_depth = 200 validation accuracy: 0.773469387755102
max_depth = 300 validation accuracy: 0.773469387755102

Best max_depth: 100
Actual depth of the trained tree: 100
Test accuracy: 0.7551020408163265


### Summary
- Training examples: **2286**
- Validation examples: **490**
- Test examples: **490**
- Best `max_depth`: **100**
- Best validation accuracy: **0.7734693878 (77.35%)**
- Test accuracy: **0.7551020408 (75.51%)**
- Actual trained-tree depth: **100**

**Important:** The exact Part (c) first-two-layer tree text is generated from the fitted model when the notebook is run. It is not fabricated in the saved-results cell.
